In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import re
import scipy.optimize as opt
from tqdm import tqdm
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr
from sympy import lambdify

In [100]:
filename = 'pareto_low_adv_refit.csv'  
t_eq=pd.read_csv('/data/zj448/SR/Ultimate_paper/pareto_archive/'+filename)

df_full = pd.read_csv('SMBH_Data_03_06_24.csv',header=1)

In [101]:
low_scatter_para=['ETG','T-type','Bar', 'Disk', 'Ring', 'Core', 'Multiple', 'Compactness', 'AGN',
       'Pseudobulge', 'BCG', 'cD','M*_sph', 'M*_gal', 'log_B/T',
       'log_sigma0', 'log_R_e_sph_maj','log_R_e_sph_eq_kpc', 'log_n_sph_maj', 'log_n_sph_eq', 'log(I_e,sph,maj/M_Sun/pc^2)',
       'log(I_e,sph,eq/M_Sun/pc^2)', 'Concentration_Index',
       'avg_Rho_1kpc_Exact_All', 'r1_density_approx', 'log10(R10_kpc)',
       'logRho_R10_approx', 'log_rho10_Exact', 'log10(R90_kpc)',
       'logRho_R90_approx', 'log_rho_90_Exact_all', 'Rho_re_spatial',
       'SR_pc_All', 'Rho_SR_pc_All', 'CR_def1_approx_new',
       'Rho_cr_def1_approx_new', 'CR_def2_approx_new',
       'Rho_CR_def2_approx_new', 'Sr(pc)_2_using_Falserm_drho',
       'Log_Approx_Avg_density_10pc', 'log_Rho_e_Exact_new',
       'logRho_e_approx_New', 'logRho_soi_approx_new',
       'log_Rho_soi_exact_new', 'Avg_Rho_Re_Exact_all',
       'Avg_Rho_soi_exact_all', 'Avg_Rho_re_Exact_all', 'Rho_re_Exact_all',
       'Rho_r_soi_2BH_approx', 'Log_Avg_Rho_10kpc_approx',
       'Log_Avg_Rho_10kpc_exact_final', 'Log_Avg_Rho_100pc_approx',
       'Log_Avg_Rho_5kpc_approx', 'Log_Avg_rho_5kpc_exact_all', 'ube', 'bve',
       'dc', 'bvtc', 'bri25', 'mabs', 'blum', 'logblum', 'logSigma0sph',
       'LogSigma0', 'R10', 'logR10', 'logR10phi', 'Rh', 'logRh', 'logRhphi',
       'logHalo','B-V','V-[3.6]','GJC23W1-W2','GJC23W2-W3','GJC23log(M*,gal/M_sun)',
       'GJC23log(SFR)','GJC23log(sSFR)','log<Sigma>_e','log<Sigma>_h','M_BH']

easy_obs_para=['LogSigma0','Concentration_Index','logSigma0sph','log_sigma0','dc','logRhphi','M*_sph','ube','bri25','bve','bvtc','logR10phi','M*_gal','log_B/T',
 'logRh','log_n_sph_eq','blum','log_R_e_sph_maj','logblum','log_n_sph_maj','logR10','Pseudobulge','AGN','Multiple','Ring','BCG','Disk','cD',
 'Bar','Core','Compactness','ETG','T-type','log10(R10_kpc)','log10(R90_kpc)','B-V','V-[3.6]','GJC23W1-W2','GJC23W2-W3','GJC23log(M*,gal/M_sun)','M_BH']

In [102]:
for para in low_scatter_para:
    if para+'_std' not in df_full.columns:
        print(para)

ETG
Bar
Disk
Ring
Core
Multiple
Compactness
AGN
Pseudobulge
BCG
cD
ube
bve
bri25
M_BH


### Assign boolean variables zero uncertainty

In [103]:
booleans=['ETG','Bar','Disk','Ring','Core','Multiple','Compactness','AGN','Pseudobulge','BCG','cD']
for b in booleans:
    df_full[b+'_std']=0

### Remove variables that do not have any uncertainties

In [104]:
## remove ube, bve, bri25
for d in ['ube','bve','bri25']:
    low_scatter_para.remove(d)
    easy_obs_para.remove(d)

In [105]:
dels=[]
for para in low_scatter_para:
    #check if all values in df_full[para+'_std'] are nan
    if para+'_std'!='M_BH_std':
        if df_full[para+'_std'].isnull().all():
            print(para)
            dels.append(para)

Concentration_Index
avg_Rho_1kpc_Exact_All
r1_density_approx
Rho_re_spatial
SR_pc_All
Rho_SR_pc_All
CR_def1_approx_new
Rho_cr_def1_approx_new
CR_def2_approx_new
Rho_CR_def2_approx_new
Sr(pc)_2_using_Falserm_drho
Avg_Rho_Re_Exact_all
Avg_Rho_soi_exact_all
Avg_Rho_re_Exact_all
Rho_re_Exact_all
Rho_r_soi_2BH_approx
B-V
V-[3.6]


In [106]:
# remove dels from low_scatter_para
for d in dels:
    try:
        low_scatter_para.remove(d)
    except:
        pass
    try:
        easy_obs_para.remove(d)
    except:
        pass

In [107]:
len(low_scatter_para),len(easy_obs_para)

(60, 35)

In [108]:
low_scatter_para

['ETG',
 'T-type',
 'Bar',
 'Disk',
 'Ring',
 'Core',
 'Multiple',
 'Compactness',
 'AGN',
 'Pseudobulge',
 'BCG',
 'cD',
 'M*_sph',
 'M*_gal',
 'log_B/T',
 'log_sigma0',
 'log_R_e_sph_maj',
 'log_R_e_sph_eq_kpc',
 'log_n_sph_maj',
 'log_n_sph_eq',
 'log(I_e,sph,maj/M_Sun/pc^2)',
 'log(I_e,sph,eq/M_Sun/pc^2)',
 'log10(R10_kpc)',
 'logRho_R10_approx',
 'log_rho10_Exact',
 'log10(R90_kpc)',
 'logRho_R90_approx',
 'log_rho_90_Exact_all',
 'Log_Approx_Avg_density_10pc',
 'log_Rho_e_Exact_new',
 'logRho_e_approx_New',
 'logRho_soi_approx_new',
 'log_Rho_soi_exact_new',
 'Log_Avg_Rho_10kpc_approx',
 'Log_Avg_Rho_10kpc_exact_final',
 'Log_Avg_Rho_100pc_approx',
 'Log_Avg_Rho_5kpc_approx',
 'Log_Avg_rho_5kpc_exact_all',
 'dc',
 'bvtc',
 'mabs',
 'blum',
 'logblum',
 'logSigma0sph',
 'LogSigma0',
 'R10',
 'logR10',
 'logR10phi',
 'Rh',
 'logRh',
 'logRhphi',
 'logHalo',
 'GJC23W1-W2',
 'GJC23W2-W3',
 'GJC23log(M*,gal/M_sun)',
 'GJC23log(SFR)',
 'GJC23log(sSFR)',
 'log<Sigma>_e',
 'log<Sigma>_h'

In [109]:
easy_obs_para

['LogSigma0',
 'logSigma0sph',
 'log_sigma0',
 'dc',
 'logRhphi',
 'M*_sph',
 'bvtc',
 'logR10phi',
 'M*_gal',
 'log_B/T',
 'logRh',
 'log_n_sph_eq',
 'blum',
 'log_R_e_sph_maj',
 'logblum',
 'log_n_sph_maj',
 'logR10',
 'Pseudobulge',
 'AGN',
 'Multiple',
 'Ring',
 'BCG',
 'Disk',
 'cD',
 'Bar',
 'Core',
 'Compactness',
 'ETG',
 'T-type',
 'log10(R10_kpc)',
 'log10(R90_kpc)',
 'GJC23W1-W2',
 'GJC23W2-W3',
 'GJC23log(M*,gal/M_sun)',
 'M_BH']

In [110]:
low_scatter_para=['ETG', 'T-type', 'Bar', 'Disk', 'Ring', 'Core', 'Multiple',
    'Compactness', 'AGN', 'Pseudobulge', 'BCG', 'cD', 
    'M*_sph', 'M*_gal', 'log_B/T', 'log_sigma0', 'log_R_e_sph_maj', 'log_R_e_sph_eq_kpc',
    'log_n_sph_maj', 'log_n_sph_eq', 'log(I_e,sph,maj/M_Sun/pc^2)', 'log(I_e,sph,eq/M_Sun/pc^2)',
    'log10(R10_kpc)', 'logRho_R10_approx', 'log_rho10_Exact', 'log10(R90_kpc)', 'logRho_R90_approx',
    'log_rho_90_Exact_all', 'Log_Approx_Avg_density_10pc', 'log_Rho_e_Exact_new', 'logRho_e_approx_New',
    'logRho_soi_approx_new', 'log_Rho_soi_exact_new', 'Log_Avg_Rho_10kpc_approx',
    'Log_Avg_Rho_10kpc_exact_final', 'Log_Avg_Rho_100pc_approx', 'Log_Avg_Rho_5kpc_approx',
    'Log_Avg_rho_5kpc_exact_all', 'dc', 'bvtc', 'mabs', 'blum', 'logblum', 'logSigma0sph',
    'LogSigma0', 'R10', 'logR10', 'logR10phi', 'Rh', 'logRh', 'logRhphi', 'logHalo', 'GJC23W1-W2',
    'GJC23W2-W3', 'GJC23log(M*,gal/M_sun)', 'GJC23log(SFR)', 'GJC23log(sSFR)', 'log<Sigma>_e',
    'log<Sigma>_h', 'M_BH']

easy_obs_para=['LogSigma0', 'logSigma0sph', 'log_sigma0', 'dc', 'logRhphi', 'M*_sph', 'bvtc',
    'logR10phi', 'M*_gal', 'log_B/T', 'logRh', 'log_n_sph_eq', 'blum', 'log_R_e_sph_maj',
    'logblum', 'log_n_sph_maj', 'logR10', 'Pseudobulge', 'AGN', 'Multiple', 'Ring', 'BCG',
    'Disk', 'cD', 'Bar', 'Core', 'Compactness', 'ETG', 'T-type', 'log10(R10_kpc)', 'log10(R90_kpc)',
    'GJC23W1-W2', 'GJC23W2-W3', 'GJC23log(M*,gal/M_sun)', 'M_BH']

In [111]:
len(low_scatter_para),len(easy_obs_para)

(60, 35)

In [112]:
from global_parameter import *

In [113]:
len(low_scatter_para),len(easy_obs_para)

(60, 35)

In [114]:
df = df_full.copy()
df_ = df[low_scatter_para].dropna(axis='index',how='any')
df = df[low_scatter_para_std]
df = df.iloc[df_.index]

In [115]:
df_

,ETG,T-type,Bar,Disk,Ring,Core,Multiple,Compactness,AGN,Pseudobulge,...,logRhphi,logHalo,GJC23W1-W2,GJC23W2-W3,"GJC23log(M*,gal/M_sun)",GJC23log(SFR),GJC23log(sSFR),log<Sigma>_e,log<Sigma>_h,M_BH
1,1,-4.8,0,0.0,0,1.0,0,0,1.0,0,...,10.782617,13.959319,-0.05,0.39,11.24,-0.763463,-12.003463,2.889338,2.939310,9.380211
3,1,-4.9,0,0.0,0,0.0,0,0,1.0,0,...,10.892139,14.060462,-0.08,0.02,11.47,-1.002614,-12.472614,2.871399,2.546708,9.102971
6,1,-2.8,0,1.0,0,0.0,0,0,0.0,0,...,9.627691,11.525458,0.03,1.28,8.85,-1.514279,-10.364279,3.640768,2.751098,5.740000
7,1,-1.2,0,1.0,0,1.0,0,0,0.0,0,...,10.908384,13.405242,-0.02,0.52,11.10,-1.059982,-12.159982,3.831612,2.884839,8.677184
9,1,-4.8,0,0.0,0,0.0,0,0,0.0,0,...,10.571088,12.727499,-0.04,0.27,10.64,-1.007889,-11.647889,2.906781,2.779434,7.591065
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,0,1.1,0,1.0,0,0.0,0,0,1.0,0,...,10.802649,13.218697,-0.02,0.90,11.06,-0.485452,-11.545452,3.480318,3.319105,8.808460
132,0,2.9,1,1.0,0,0.0,0,0,0.0,1,...,10.883571,13.233744,0.00,2.20,11.06,0.202434,-10.857566,3.245399,3.576320,8.279168
133,0,2.3,0,1.0,1,0.0,0,0,1.0,1,...,10.535837,12.289895,0.02,2.71,10.38,-0.082862,-10.462862,4.462654,3.755643,6.833669
134,0,2.2,0,1.0,0,0.0,0,0,1.0,1,...,10.477966,12.536387,0.02,2.21,10.54,-0.141944,-10.681944,3.729387,3.316511,6.198130


In [116]:
for para in low_scatter_para:
    #print how many nans in para+'_std'
    try:
        if np.isnan(df[para+'_std']).sum()>0:
            print(para,np.isnan(df[para+'_std']).sum())
    except:
        pass

log_n_sph_eq 66


### further remove 'log_n_sph_eq' and 'log(I_e,sph,eq/M_Sun/pc^2)'

In [118]:
# removed in global_parameters
from global_parameter import *
len(low_scatter_para),len(easy_obs_para)

(58, 34)

In [120]:
df_full[low_scatter_para_std].dropna(axis='index',how='any')

,ETG,T-type,Bar,Disk,Ring,Core,Multiple,Compactness,AGN,Pseudobulge,...,logRhphi_std,logHalo_std,GJC23W1-W2_std,GJC23W2-W3_std,"GJC23log(M*,gal/M_sun)_std",GJC23log(SFR)_std,GJC23log(sSFR)_std,log<Sigma>_e_std,log<Sigma>_h_std,M_BH_std_sym
1,1,-4.8,0,0.0,0,1.0,0,0,1.0,0,...,0.170524,0.799963,0.04,0.06,0.08,0.066001,0.103712,0.128944,0.128944,0.180956
3,1,-4.9,0,0.0,0,0.0,0,0,1.0,0,...,0.141405,0.807545,0.04,0.08,0.08,0.079082,0.112490,0.094548,0.094548,0.086075
6,1,-2.8,0,1.0,0,0.0,0,0,0.0,0,...,0.160544,0.760591,0.04,0.05,0.09,0.130572,0.158585,0.123588,0.123588,0.103967
7,1,-1.2,0,1.0,0,1.0,0,0,0.0,0,...,0.130661,0.766852,0.04,0.06,0.08,0.055845,0.097564,0.073712,0.073712,0.099526
9,1,-4.8,0,0.0,0,0.0,0,0,0.0,0,...,0.152391,0.743872,0.04,0.13,0.08,0.073414,0.108580,0.093730,0.093730,0.194876
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,0,1.1,0,1.0,0,0.0,0,0,1.0,0,...,0.170416,0.758740,0.04,0.05,0.08,0.043164,0.090902,0.079789,0.079789,0.027143
132,0,2.9,1,1.0,0,0.0,0,0,0.0,1,...,0.236217,0.759340,0.04,0.04,0.08,0.043135,0.090888,0.175905,0.175905,0.052000
133,0,2.3,0,1.0,1,0.0,0,0,1.0,1,...,0.143617,0.737455,0.04,0.04,0.08,0.042152,0.090426,0.097945,0.097945,0.101335
134,0,2.2,0,1.0,0,0.0,0,0,1.0,1,...,0.150851,0.740322,0.04,0.04,0.08,0.042032,0.090370,0.109068,0.109068,0.108574


In [121]:
df_full[easy_obs_para_std].dropna(axis='index',how='any')

,LogSigma0,logSigma0sph,log_sigma0,dc,logRhphi,M*_sph,bvtc,logR10phi,M*_gal,log_B/T,...,Core_std,Compactness_std,ETG_std,T-type_std,log10(R10_kpc)_std,log10(R90_kpc)_std,GJC23W1-W2_std,GJC23W2-W3_std,"GJC23log(M*,gal/M_sun)_std",M_BH_std_sym
1,4.704516,8.218084,2.471453,37.519414,10.782617,11.69,0.947,10.148833,11.69,0.00,...,0,0,0,0.4,0.122041,0.122041,0.04,0.06,0.08,0.180956
3,4.033710,5.486389,2.515012,65.993965,10.892139,11.72,0.912,10.470194,11.74,-0.02,...,0,0,0,0.3,0.078167,0.078167,0.04,0.08,0.08,0.086075
6,4.008665,4.025433,1.538951,3.253966,9.627691,8.03,0.883,9.346477,9.19,-1.16,...,0,0,0,0.7,0.065420,0.065420,0.04,0.05,0.09,0.103967
7,4.059098,3.648904,2.374180,28.955108,10.908384,10.88,0.946,10.302046,11.38,-0.50,...,0,0,0,0.6,0.066468,0.066468,0.04,0.06,0.08,0.099526
9,4.179855,7.738421,2.296073,19.340791,10.571088,10.84,0.865,9.894541,10.90,-0.06,...,0,0,0,0.4,0.080682,0.080682,0.04,0.13,0.08,0.194876
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,4.779024,6.191939,2.353474,24.032183,10.802649,11.04,0.878,10.006116,11.26,-0.22,...,0,0,0,0.4,0.073080,0.073080,0.04,0.05,0.08,0.027143
132,4.714507,8.242104,2.283934,23.953960,10.883571,10.30,0.825,10.014525,11.27,-0.97,...,0,0,0,0.4,0.157888,0.157888,0.04,0.04,0.08,0.052000
133,4.836918,4.922064,2.030762,11.281387,10.535837,10.03,0.718,9.564854,10.51,-0.48,...,0,0,0,0.8,0.070204,0.070204,0.04,0.04,0.08,0.101335
134,4.798361,4.021619,1.987443,22.347003,10.477966,9.88,0.742,9.718380,10.74,-0.86,...,0,0,0,0.6,0.069536,0.069536,0.04,0.04,0.08,0.108574
